In [6]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from joblib import Parallel, delayed
from itertools import combinations

In [7]:
lam, p, niter = 1e4, 0.01, 10

def baseline_als(y):
    L = len(y)
    D = np.diff(np.eye(L), 2)
    D = lam * D.dot(D.T)
    w = np.ones(L)
    for _ in range(niter):
        b = np.linalg.solve(np.diag(w) + D, w * y)
        w = p * (y > b) + (1 - p) * (y < b)
    return b

def preprocess(arr):
    """
    Baseline-correct → first derivative → L2-normalize (batch).
    """
    out = np.zeros_like(arr)
    for i, s in enumerate(arr):
        b = baseline_als(s)
        c = s - b
        d = np.gradient(c)              # first derivative
        norm = np.linalg.norm(d)
        out[i] = d / norm if norm > 0 else d
    return out

def preprocess_single(spectrum):
    """
    Baseline-correct → first derivative → L2-normalize (single spectrum).
    """
    b = baseline_als(spectrum)
    c = spectrum - b
    d = np.gradient(c)                  # first derivative
    norm = np.linalg.norm(d)
    out = d / norm if norm > 0 else d
    return out

def floatify_cols(df):
    new = []
    for c in df.columns:
        if c in ('Label', 'Label 1', 'Label 2'):
            new.append(c)
        else:
            new.append(float(c))
    df.columns = new

In [8]:
# ─── 1) Load reference_v2 and preprocess ──────────────────────────────────────
ref_df = pd.read_csv('reference_v2.csv')

floatify_cols(ref_df)
wav_cols = [c for c in ref_df.columns if c != 'Label']
ref_specs  = ref_df[wav_cols].values       # (n_ref_samples, n_waves)
ref_labels = ref_df['Label'].values        # (n_ref_samples,)

In [9]:
classes    = sorted(np.unique(ref_labels))
C = len(classes)
class_to_i = {c:i for i,c in enumerate(classes)}

In [10]:
# ─── 2) Generate synthetic mixtures on RAW spectra ────────────────────────────
ratios = np.arange(0.05, 1.0, 0.05)
noise_level = 0.01
n_per_ratio = 10  # number of random spectra per pair/ratio

synth_specs = []
synth_labels = []
for (i, ci), (j, cj) in combinations(enumerate(classes), 2):
    idx_i = np.where(ref_labels == ci)[0]
    idx_j = np.where(ref_labels == cj)[0]
    for r in ratios:
        for _ in range(n_per_ratio):
            spec_i = ref_specs[np.random.choice(idx_i)]
            spec_j = ref_specs[np.random.choice(idx_j)]
            mix = r * spec_i + (1-r) * spec_j
            mix += np.random.normal(scale=noise_level, size=mix.shape)
            synth_specs.append(mix)
            synth_labels.append((ci, cj))
synth_specs = np.array(synth_specs)        # (n_synth, n_waves)
print("Synthetic raw spectra:", synth_specs.shape)

Synthetic raw spectra: (12540, 1024)


In [11]:
# ─── 3) Apply derivative-based preprocessing to synthetic spectra ─────────────
synth_proc = np.vstack(
    Parallel(n_jobs=-1, verbose=10)(
        delayed(preprocess_single)(spec) 
        for spec in synth_specs
    )
)
print("Parallel preprocess (derivative) done:", synth_proc.shape)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    1.9s
[Parallel(n_jobs=-1)]: Done  49 tasks      | elapsed:    2.1s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done  81 tasks      | elapsed:    3.1s
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed:    3.8s
[Parallel(n_jobs=-1)]: Done 117 tasks      | elapsed:    4.2s
[Parallel(n_jobs=-1)]: Done 136 tasks      | elapsed:    4.9s
[Parallel(n_jobs=-1)]: Done 157 tasks      | elapsed:    5.4s
[Parallel(n_jobs=-1)]: Done 178 tasks      | elapsed:    6.1s
[Parallel(n_jobs=-1)]: Done 201 tasks      | elapsed:    6.9s
[Parallel(n_jobs=-1)]: Done 224 tasks      | elapsed:    7.5s
[Parallel(n_jobs=-1)]: Done 249 tasks      | elapsed:    8.3s
[Parallel(n_jobs=-1)]: Done 274 tasks      | elapsed:  

Parallel preprocess (derivative) done: (12540, 1024)


[Parallel(n_jobs=-1)]: Done 12540 out of 12540 | elapsed:  6.4min finished


In [12]:
X_train, X_test, y_train_pairs, y_test_pairs = train_test_split(
    synth_proc, synth_labels, test_size=0.2, random_state=42
)

In [13]:
# ─── 5) Dataset for contrastive learning ──────────────────────────────────────
class RamanPairDataset(Dataset):
    def __init__(self, specs, pair_labels, augment_fn=None):
        self.specs  = specs
        self.labels = pair_labels
        self.augment = augment_fn
        self.by_label = {}
        for idx, lab in enumerate(pair_labels):
            self.by_label.setdefault(lab, []).append(idx)

    def __len__(self):
        return len(self.specs)

    def __getitem__(self, idx):
        import random
        x1 = self.specs[idx]
        lab1 = self.labels[idx]

        if random.random() < 0.5:
            # positive pair
            pos_idx = random.choice(self.by_label[lab1])
            x2 = self.specs[pos_idx]
            y = 1.0
        else:
            # negative pair
            neg_classes = [l for l in self.by_label.keys() if l != lab1]
            neg_lab = random.choice(neg_classes)
            neg_idx = random.choice(self.by_label[neg_lab])
            x2 = self.specs[neg_idx]
            y = 0.0

        if self.augment:
            x1 = self.augment(x1)
            x2 = self.augment(x2)

        return (
            torch.tensor(x1, dtype=torch.float32).unsqueeze(0),
            torch.tensor(x2, dtype=torch.float32).unsqueeze(0),
            torch.tensor(y,  dtype=torch.float32)
        )

In [14]:
def augment(spec, noise_std=0.01, shift_max=2):
    spec_noisy = spec + np.random.normal(0, noise_std, size=spec.shape)
    shift = np.random.randint(-shift_max, shift_max + 1)
    return np.roll(spec_noisy, shift)

train_ds = RamanPairDataset(X_train, y_train_pairs, augment_fn=augment)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

In [15]:
# ─── 6) Siamese network & contrastive loss ────────────────────────────────────
class SiameseNet(nn.Module):
    def __init__(self, input_len, embed_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1,16,7,padding=3), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16,32,5,padding=2), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Flatten(),
            nn.Linear((input_len//4)*32, embed_dim),
            nn.ReLU()
        )
    def forward(self,x):
        z = self.encoder(x)
        return F.normalize(z, dim=1)

def contrastive_loss(z1, z2, label, margin=1.0):
    dist = F.pairwise_distance(z1, z2)
    return (label*dist**2 + (1-label)*F.relu(margin-dist)**2).mean()

In [16]:
# ─── 7) Train the Siamese on derivative-processed mixtures ────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SiameseNet(input_len=synth_proc.shape[1], embed_dim=64).to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)

In [20]:
for epoch in range(1, 21):
    model.train()
    total = 0.0
    for x1, x2, l in train_loader:
        x1,x2,l = x1.to(device), x2.to(device), l.to(device)
        z1, z2 = model(x1), model(x2)
        loss = contrastive_loss(z1, z2, l)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()*x1.size(0)
    print(f"Epoch {epoch:03d} Loss: {total/len(train_ds):.4f}")

Epoch 001 Loss: 0.0460
Epoch 002 Loss: 0.0483
Epoch 003 Loss: 0.0462
Epoch 004 Loss: 0.0483
Epoch 005 Loss: 0.0477
Epoch 006 Loss: 0.0456
Epoch 007 Loss: 0.0486
Epoch 008 Loss: 0.0465
Epoch 009 Loss: 0.0471
Epoch 010 Loss: 0.0451
Epoch 011 Loss: 0.0471
Epoch 012 Loss: 0.0464
Epoch 013 Loss: 0.0438
Epoch 014 Loss: 0.0471
Epoch 015 Loss: 0.0461
Epoch 016 Loss: 0.0470
Epoch 017 Loss: 0.0462
Epoch 018 Loss: 0.0486
Epoch 019 Loss: 0.0437
Epoch 020 Loss: 0.0444


In [21]:
torch.save(model.state_dict(), "siamese_mixture_deriv.pth")